[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/xla/lab-x6-collectives-on-a-mock-gpu.ipynb)

# LAB·X6 · Collectives on a mock GPU

**Hardware:** any machine. Everything below runs on CPU.

Where does a runtime implement collectives? Open the PJRT C API and search for all-reduce: there is no such function. Nothing in the interface knows what a collective is, and yet every distributed JAX program depends on them. This lab resolves that tension by building the runtime side yourself: LAB·X5's mock plugin, grown from one device to eight GPU-shaped ones, so that sharded arrays, `shard_map`, and `psum` all flow through code you compiled. You will watch a collective arrive at the plugin as an instruction inside the program, not as an API call, and you will stand at the exact function where a real GPU plugin would run it.

LAB·X5 is the prerequisite; this notebook repeats none of its explanations.

In [ ]:
# Same pinned pair as LAB·X5; the header below comes from the XLA commit
# this jaxlib was built from.
%pip install -q "jax==0.4.38" "jaxlib==0.4.38"
import jax
print("jax", jax.__version__)

## From one device to eight

The delta from LAB·X5's plugin is almost entirely bookkeeping, which is itself the lesson: nothing about multi-device changes the shape of the interface. The singleton device, description, and memory become arrays of eight, initialized in a loop. Each buffer records which device it lives on. `PJRT_Client_DefaultDeviceAssignment` fills its grid by cycling through device ids, which is the answer JAX gets when it asks "I have N replicas and M partitions, which devices do I use?". And the platform now calls itself `mockgpu` with device kind `mock-gpu`, because nothing but those strings is what "being a GPU" means to the interface.

**your prediction:** `jax.device_put` of one `(8, 4)` array, sharded eight ways along the first axis, reaches the plugin as how many calls?

In [ ]:
import pathlib, urllib.request

XLA_COMMIT = "20a482597b7dd3067b26ca382b88084ee5a21cf7"  # pinned by jax v0.4.38
hdr_dir = pathlib.Path("include/xla/pjrt/c")
hdr_dir.mkdir(parents=True, exist_ok=True)
url = f"https://raw.githubusercontent.com/openxla/xla/{XLA_COMMIT}/xla/pjrt/c/pjrt_c_api.h"
urllib.request.urlretrieve(url, hdr_dir / "pjrt_c_api.h")
print("header fetched")

In [ ]:
mock_gpu_c = r'''// mock_gpu.c: a GPU-shaped mock PJRT plugin with eight devices.
//
// One platform ("mockgpu"), eight devices, one memory space per device.
// Compile accepts the program and stores it, so sharded programs and their
// collectives flow through untouched. Buffers are per-device host memory,
// so an eight-way sharded array becomes eight real buffers on eight mock
// devices. Execute refuses: in a real GPU plugin, that is exactly where
// collectives run (the all-reduce thunk calling NCCL). Here, nothing runs.
//
// Build:  cc -shared -fPIC -I <xla-src-root> mock_gpu.c -o mock_gpu.so

#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#include "xla/pjrt/c/pjrt_c_api.h"

// ---------------------------------------------------------------- errors ----
// The C API reports failure by returning a PJRT_Error*; NULL means success.
// The framework asks a failed call for its message and code, then destroys it.

struct PJRT_Error {
  PJRT_Error_Code code;
  char message[512];
};

static PJRT_Error* make_error(PJRT_Error_Code code, const char* msg) {
  PJRT_Error* err = (PJRT_Error*)malloc(sizeof(PJRT_Error));
  err->code = code;
  snprintf(err->message, sizeof(err->message), "%s", msg);
  return err;
}

static void Error_Destroy(PJRT_Error_Destroy_Args* args) {
  free(args->error);
}

static void Error_Message(PJRT_Error_Message_Args* args) {
  args->message = args->error->message;
  args->message_size = strlen(args->error->message);
}

static PJRT_Error* Error_GetCode(PJRT_Error_GetCode_Args* args) {
  args->code = args->error->code;
  return NULL;
}

#define UNIMPLEMENTED(name)                                        \
  make_error(PJRT_Error_Code_UNIMPLEMENTED,                        \
             "mock PJRT plugin: " name " is not implemented; this " \
             "device compiles but does not execute")

// ---------------------------------------------------------------- objects ---
// The header only forward-declares these structs; the plugin owns their
// layout. One static instance each: this plugin is one device, full stop.

#define MOCK_NUM_DEVICES 8

struct PJRT_DeviceDescription {
  int id;
  char debug_string[32];
  char to_string[32];
};
struct PJRT_Memory {
  int id;
  char debug_string[32];
  char to_string[32];
  struct PJRT_Device* device;
};
struct PJRT_Device {
  struct PJRT_DeviceDescription* description;
  struct PJRT_Memory* memory;
};
struct PJRT_Client {
  int dummy;
};
struct PJRT_Executable {
  char* program;        // the bytes Compile was handed, verbatim
  size_t program_size;
  char format[16];      // "mlir" or "hlo", whatever the caller said it was
  int ref_count;
};
struct PJRT_LoadedExecutable {
  struct PJRT_Executable* executable;
  struct PJRT_Client* client;
};

static struct PJRT_DeviceDescription mock_descriptions[MOCK_NUM_DEVICES];
static struct PJRT_Memory mock_memory_objs[MOCK_NUM_DEVICES];
static struct PJRT_Device mock_device_objs[MOCK_NUM_DEVICES];
static struct PJRT_Client mock_client;
static PJRT_Device* mock_devices[MOCK_NUM_DEVICES];
static PJRT_Memory* mock_memories[MOCK_NUM_DEVICES];

static void init_topology(void) {
  for (int i = 0; i < MOCK_NUM_DEVICES; ++i) {
    mock_descriptions[i].id = i;
    snprintf(mock_descriptions[i].debug_string,
             sizeof(mock_descriptions[i].debug_string), "MockGpu(id=%d)", i);
    snprintf(mock_descriptions[i].to_string,
             sizeof(mock_descriptions[i].to_string), "mockgpu:%d", i);
    mock_memory_objs[i].id = i;
    snprintf(mock_memory_objs[i].debug_string,
             sizeof(mock_memory_objs[i].debug_string), "MockHbm(id=%d)", i);
    snprintf(mock_memory_objs[i].to_string,
             sizeof(mock_memory_objs[i].to_string), "mock_hbm:%d", i);
    mock_memory_objs[i].device = &mock_device_objs[i];
    mock_device_objs[i].description = &mock_descriptions[i];
    mock_device_objs[i].memory = &mock_memory_objs[i];
    mock_devices[i] = &mock_device_objs[i];
    mock_memories[i] = &mock_memory_objs[i];
  }
}

// ----------------------------------------------------------------- plugin ---

static PJRT_Error* Plugin_Initialize(PJRT_Plugin_Initialize_Args* args) {
  (void)args;
  return NULL;
}

static PJRT_Error* Plugin_Attributes(PJRT_Plugin_Attributes_Args* args) {
  args->num_attributes = 0;
  args->attributes = NULL;
  return NULL;
}

// ----------------------------------------------------------------- client ---

static PJRT_Error* Client_Create(PJRT_Client_Create_Args* args) {
  args->client = &mock_client;
  return NULL;
}

static PJRT_Error* Client_Destroy(PJRT_Client_Destroy_Args* args) {
  (void)args;  // everything is static; nothing to free
  return NULL;
}

static PJRT_Error* Client_PlatformName(PJRT_Client_PlatformName_Args* args) {
  args->platform_name = "mockgpu";
  args->platform_name_size = 7;
  return NULL;
}

static PJRT_Error* Client_ProcessIndex(PJRT_Client_ProcessIndex_Args* args) {
  args->process_index = 0;
  return NULL;
}

static PJRT_Error* Client_PlatformVersion(
    PJRT_Client_PlatformVersion_Args* args) {
  args->platform_version = "mockgpu 0.1";
  args->platform_version_size = 11;
  return NULL;
}

static PJRT_Error* Client_Devices(PJRT_Client_Devices_Args* args) {
  args->devices = mock_devices;
  args->num_devices = MOCK_NUM_DEVICES;
  return NULL;
}

static PJRT_Error* Client_AddressableDevices(
    PJRT_Client_AddressableDevices_Args* args) {
  args->addressable_devices = mock_devices;
  args->num_addressable_devices = MOCK_NUM_DEVICES;
  return NULL;
}

static PJRT_Error* Client_LookupDevice(PJRT_Client_LookupDevice_Args* args) {
  if (args->id < 0 || args->id >= MOCK_NUM_DEVICES)
    return make_error(PJRT_Error_Code_NOT_FOUND, "no such mock device id");
  args->device = mock_devices[args->id];
  return NULL;
}

static PJRT_Error* Client_LookupAddressableDevice(
    PJRT_Client_LookupAddressableDevice_Args* args) {
  if (args->local_hardware_id < 0 ||
      args->local_hardware_id >= MOCK_NUM_DEVICES)
    return make_error(PJRT_Error_Code_NOT_FOUND, "no such mock device id");
  args->addressable_device = mock_devices[args->local_hardware_id];
  return NULL;
}

static PJRT_Error* Client_AddressableMemories(
    PJRT_Client_AddressableMemories_Args* args) {
  args->addressable_memories = mock_memories;
  args->num_addressable_memories = MOCK_NUM_DEVICES;
  return NULL;
}

static PJRT_Error* Client_DefaultDeviceAssignment(
    PJRT_Client_DefaultDeviceAssignment_Args* args) {
  for (size_t i = 0; i < args->default_assignment_size; ++i) {
    args->default_assignment[i] = (int)(i % MOCK_NUM_DEVICES);
  }
  return NULL;
}

static PJRT_Error* Client_TopologyDescription(
    PJRT_Client_TopologyDescription_Args* args) {
  (void)args;
  return UNIMPLEMENTED("PJRT_Client_TopologyDescription");
}

// ----------------------------------------------------- device description ---

static PJRT_Error* DeviceDescription_Id(PJRT_DeviceDescription_Id_Args* args) {
  args->id = args->device_description->id;
  return NULL;
}

static PJRT_Error* DeviceDescription_ProcessIndex(
    PJRT_DeviceDescription_ProcessIndex_Args* args) {
  args->process_index = 0;
  return NULL;
}

static PJRT_Error* DeviceDescription_Attributes(
    PJRT_DeviceDescription_Attributes_Args* args) {
  args->num_attributes = 0;
  args->attributes = NULL;
  return NULL;
}

static PJRT_Error* DeviceDescription_Kind(
    PJRT_DeviceDescription_Kind_Args* args) {
  args->device_kind = "mock-gpu";
  args->device_kind_size = 8;
  return NULL;
}

static PJRT_Error* DeviceDescription_DebugString(
    PJRT_DeviceDescription_DebugString_Args* args) {
  args->debug_string = args->device_description->debug_string;
  args->debug_string_size = strlen(args->device_description->debug_string);
  return NULL;
}

static PJRT_Error* DeviceDescription_ToString(
    PJRT_DeviceDescription_ToString_Args* args) {
  args->to_string = args->device_description->to_string;
  args->to_string_size = strlen(args->device_description->to_string);
  return NULL;
}

// ----------------------------------------------------------------- device ---

static PJRT_Error* Device_GetDescription(PJRT_Device_GetDescription_Args* args) {
  args->device_description = args->device->description;
  return NULL;
}

static PJRT_Error* Device_IsAddressable(PJRT_Device_IsAddressable_Args* args) {
  args->is_addressable = true;
  return NULL;
}

static PJRT_Error* Device_LocalHardwareId(
    PJRT_Device_LocalHardwareId_Args* args) {
  args->local_hardware_id = args->device->description->id;
  return NULL;
}

static PJRT_Error* Device_AddressableMemories(
    PJRT_Device_AddressableMemories_Args* args) {
  args->memories = &args->device->memory;
  args->num_memories = 1;
  return NULL;
}

static PJRT_Error* Device_DefaultMemory(PJRT_Device_DefaultMemory_Args* args) {
  args->memory = args->device->memory;
  return NULL;
}

static PJRT_Error* Device_MemoryStats(PJRT_Device_MemoryStats_Args* args) {
  (void)args;
  return UNIMPLEMENTED("PJRT_Device_MemoryStats");
}

// ----------------------------------------------------------------- memory ---

static PJRT_Error* Memory_Id(PJRT_Memory_Id_Args* args) {
  args->id = args->memory->id;
  return NULL;
}

static PJRT_Error* Memory_Kind(PJRT_Memory_Kind_Args* args) {
  args->kind = "mock_hbm";
  args->kind_size = strlen("mock_hbm");
  return NULL;
}

static PJRT_Error* Memory_KindId(PJRT_Memory_Kind_Id_Args* args) {
  args->kind_id = 0;
  return NULL;
}

static PJRT_Error* Memory_DebugString(PJRT_Memory_DebugString_Args* args) {
  args->debug_string = args->memory->debug_string;
  args->debug_string_size = strlen(args->memory->debug_string);
  return NULL;
}

static PJRT_Error* Memory_ToString(PJRT_Memory_ToString_Args* args) {
  args->to_string = args->memory->to_string;
  args->to_string_size = strlen(args->memory->to_string);
  return NULL;
}

static PJRT_Error* Memory_AddressableByDevices(
    PJRT_Memory_AddressableByDevices_Args* args) {
  args->devices = mock_devices;
  args->num_devices = MOCK_NUM_DEVICES;
  return NULL;
}

// ---------------------------------------------------------------- compile ---
// The one promise this plugin keeps for real: accept the program. The bytes
// JAX sends are StableHLO as MLIR bytecode; we store them verbatim and hand
// them back if asked for the "optimized" program. Compilation, here, is the
// identity function. Everything the interface requires still works.

static PJRT_Error* Client_Compile(PJRT_Client_Compile_Args* args) {
  struct PJRT_Executable* exe =
      (struct PJRT_Executable*)malloc(sizeof(struct PJRT_Executable));
  exe->program_size = args->program->code_size;
  exe->program = (char*)malloc(exe->program_size);
  memcpy(exe->program, args->program->code, exe->program_size);
  snprintf(exe->format, sizeof(exe->format), "%.*s",
           (int)args->program->format_size, args->program->format);
  exe->ref_count = 1;

  struct PJRT_LoadedExecutable* loaded =
      (struct PJRT_LoadedExecutable*)malloc(
          sizeof(struct PJRT_LoadedExecutable));
  loaded->executable = exe;
  loaded->client = args->client;
  args->executable = loaded;
  return NULL;
}

static PJRT_Error* Executable_Destroy(PJRT_Executable_Destroy_Args* args) {
  struct PJRT_Executable* exe = args->executable;
  if (--exe->ref_count == 0) {
    free(exe->program);
    free(exe);
  }
  return NULL;
}

static PJRT_Error* LoadedExecutable_Destroy(
    PJRT_LoadedExecutable_Destroy_Args* args) {
  struct PJRT_Executable* exe = args->executable->executable;
  if (--exe->ref_count == 0) {
    free(exe->program);
    free(exe);
  }
  free(args->executable);
  return NULL;
}

static PJRT_Error* LoadedExecutable_GetExecutable(
    PJRT_LoadedExecutable_GetExecutable_Args* args) {
  args->loaded_executable->executable->ref_count++;
  args->executable = args->loaded_executable->executable;
  return NULL;
}

static PJRT_Error* Executable_Name(PJRT_Executable_Name_Args* args) {
  args->executable_name = "mock_executable";
  args->executable_name_size = strlen("mock_executable");
  return NULL;
}

static PJRT_Error* Executable_NumReplicas(
    PJRT_Executable_NumReplicas_Args* args) {
  args->num_replicas = 1;
  return NULL;
}

static PJRT_Error* Executable_NumPartitions(
    PJRT_Executable_NumPartitions_Args* args) {
  args->num_partitions = 1;
  return NULL;
}

static PJRT_Error* LoadedExecutable_AddressableDevices(
    PJRT_LoadedExecutable_AddressableDevices_Args* args) {
  args->addressable_devices = mock_devices;
  args->num_addressable_devices = MOCK_NUM_DEVICES;
  return NULL;
}

static PJRT_Error* LoadedExecutable_Delete(
    PJRT_LoadedExecutable_Delete_Args* args) {
  (void)args;
  return NULL;
}

static PJRT_Error* LoadedExecutable_IsDeleted(
    PJRT_LoadedExecutable_IsDeleted_Args* args) {
  args->is_deleted = false;
  return NULL;
}

static PJRT_Error* Executable_OptimizedProgram(
    PJRT_Executable_OptimizedProgram_Args* args) {
  struct PJRT_Executable* exe = args->executable;
  PJRT_Program* out = args->program;
  out->format = exe->format;
  out->format_size = strlen(exe->format);
  if (out->code == NULL) {
    // First call: report the size. Second call: copy the bytes.
    out->code_size = exe->program_size;
    return NULL;
  }
  memcpy(out->code, exe->program, exe->program_size);
  return NULL;
}

static PJRT_Error* Executable_NumOutputs(PJRT_Executable_NumOutputs_Args* args) {
  args->num_outputs = 1;
  return NULL;
}

static PJRT_Error* Executable_SizeOfGeneratedCodeInBytes(
    PJRT_Executable_SizeOfGeneratedCodeInBytes_Args* args) {
  args->size_in_bytes = (int64_t)args->executable->program_size;
  return NULL;
}

static PJRT_Error* Executable_GetCostAnalysis(
    PJRT_Executable_GetCostAnalysis_Args* args) {
  (void)args;
  return UNIMPLEMENTED("PJRT_Executable_GetCostAnalysis");
}

static PJRT_Error* Executable_OutputMemoryKinds(
    PJRT_Executable_OutputMemoryKinds_Args* args) {
  (void)args;
  return UNIMPLEMENTED("PJRT_Executable_OutputMemoryKinds");
}

static PJRT_Error* Executable_OutputElementTypes(
    PJRT_Executable_OutputElementTypes_Args* args) {
  (void)args;
  return UNIMPLEMENTED("PJRT_Executable_OutputElementTypes");
}

static PJRT_Error* Executable_OutputDimensions(
    PJRT_Executable_OutputDimensions_Args* args) {
  (void)args;
  return UNIMPLEMENTED("PJRT_Executable_OutputDimensions");
}

static PJRT_Error* Executable_Fingerprint(
    PJRT_Executable_Fingerprint_Args* args) {
  args->executable_fingerprint = "mock-fingerprint";
  args->executable_fingerprint_size = strlen("mock-fingerprint");
  return NULL;
}

// ----------------------------------------------------------------- events ---
// Everything this plugin does finishes before the call returns, so every
// event is born ready. The framework still expects the full event protocol.

struct PJRT_Event {
  int dummy;
};

static PJRT_Error* Event_Destroy(PJRT_Event_Destroy_Args* args) {
  free(args->event);
  return NULL;
}

static PJRT_Error* Event_IsReady(PJRT_Event_IsReady_Args* args) {
  args->is_ready = true;
  return NULL;
}

static PJRT_Error* Event_Error(PJRT_Event_Error_Args* args) {
  (void)args;
  return NULL;  // a ready event with no error
}

static PJRT_Error* Event_Await(PJRT_Event_Await_Args* args) {
  (void)args;
  return NULL;
}

static PJRT_Error* Event_OnReady(PJRT_Event_OnReady_Args* args) {
  args->callback(NULL, args->user_arg);  // already ready: fire immediately
  return NULL;
}

static struct PJRT_Event* make_ready_event(void) {
  return (struct PJRT_Event*)calloc(1, sizeof(struct PJRT_Event));
}

// ---------------------------------------------------------------- buffers ---
// A PjRtBuffer is memory on one device. This device's memory is plain host
// memory the plugin mallocs, which is the honest version of the lesson: the
// interface never says what a device is, only who owns the bytes.

#define MOCK_MAX_DIMS 16

struct PJRT_Buffer {
  struct PJRT_Device* device;
  char* data;
  size_t size_bytes;
  PJRT_Buffer_Type type;
  int64_t dims[MOCK_MAX_DIMS];
  int64_t minor_to_major[MOCK_MAX_DIMS];
  size_t num_dims;
  bool deleted;
};

static const size_t mock_elem_size[] = {
    [PJRT_Buffer_Type_PRED] = 1,  [PJRT_Buffer_Type_S8] = 1,
    [PJRT_Buffer_Type_S16] = 2,   [PJRT_Buffer_Type_S32] = 4,
    [PJRT_Buffer_Type_S64] = 8,   [PJRT_Buffer_Type_U8] = 1,
    [PJRT_Buffer_Type_U16] = 2,   [PJRT_Buffer_Type_U32] = 4,
    [PJRT_Buffer_Type_U64] = 8,   [PJRT_Buffer_Type_F16] = 2,
    [PJRT_Buffer_Type_F32] = 4,   [PJRT_Buffer_Type_F64] = 8,
    [PJRT_Buffer_Type_BF16] = 2,  [PJRT_Buffer_Type_C64] = 8,
    [PJRT_Buffer_Type_C128] = 16,
};

static PJRT_Error* Client_BufferFromHostBuffer(
    PJRT_Client_BufferFromHostBuffer_Args* args) {
  if (args->num_dims > MOCK_MAX_DIMS)
    return make_error(PJRT_Error_Code_UNIMPLEMENTED,
                      "mock PJRT plugin: too many dimensions");
  if (args->type >= sizeof(mock_elem_size) / sizeof(mock_elem_size[0]) ||
      mock_elem_size[args->type] == 0)
    return make_error(PJRT_Error_Code_UNIMPLEMENTED,
                      "mock PJRT plugin: unsupported element type");
  if (args->byte_strides != NULL && args->num_byte_strides != 0) {
    // Accept only dense row-major strides; anything else stays unimplemented.
    int64_t expect = (int64_t)mock_elem_size[args->type];
    for (size_t i = args->num_dims; i > 0; --i) {
      if (args->byte_strides[i - 1] != expect)
        return UNIMPLEMENTED("non-row-major byte_strides");
      expect *= args->dims[i - 1];
    }
  }

  struct PJRT_Buffer* buf =
      (struct PJRT_Buffer*)calloc(1, sizeof(struct PJRT_Buffer));
  buf->device = args->device != NULL  ? args->device
               : args->memory != NULL ? args->memory->device
                                      : mock_devices[0];
  buf->type = args->type;
  buf->num_dims = args->num_dims;
  size_t n = mock_elem_size[args->type];
  for (size_t i = 0; i < args->num_dims; ++i) {
    buf->dims[i] = args->dims[i];
    buf->minor_to_major[i] = (int64_t)(args->num_dims - 1 - i);
    n *= (size_t)args->dims[i];
  }
  buf->size_bytes = n;
  buf->data = (char*)malloc(n);
  memcpy(buf->data, args->data, n);  // "transfer" to the device: one memcpy

  args->buffer = buf;
  args->done_with_host_buffer = make_ready_event();
  return NULL;
}

static PJRT_Error* Buffer_Destroy(PJRT_Buffer_Destroy_Args* args) {
  free(args->buffer->data);
  free(args->buffer);
  return NULL;
}

static PJRT_Error* Buffer_ElementType(PJRT_Buffer_ElementType_Args* args) {
  args->type = args->buffer->type;
  return NULL;
}

static PJRT_Error* Buffer_Dimensions(PJRT_Buffer_Dimensions_Args* args) {
  args->dims = args->buffer->dims;
  args->num_dims = args->buffer->num_dims;
  return NULL;
}

static PJRT_Error* Buffer_UnpaddedDimensions(
    PJRT_Buffer_UnpaddedDimensions_Args* args) {
  args->unpadded_dims = args->buffer->dims;
  args->num_dims = args->buffer->num_dims;
  return NULL;
}

static PJRT_Error* Buffer_DynamicDimensionIndices(
    PJRT_Buffer_DynamicDimensionIndices_Args* args) {
  args->dynamic_dim_indices = NULL;
  args->num_dynamic_dims = 0;
  return NULL;
}

static PJRT_Error* Buffer_GetMemoryLayout(
    PJRT_Buffer_GetMemoryLayout_Args* args) {
  args->layout.type = PJRT_Buffer_MemoryLayout_Type_Tiled;
  args->layout.tiled.minor_to_major = args->buffer->minor_to_major;
  args->layout.tiled.minor_to_major_size = args->buffer->num_dims;
  args->layout.tiled.tile_dims = NULL;
  args->layout.tiled.tile_dim_sizes = NULL;
  args->layout.tiled.num_tiles = 0;
  return NULL;
}

static PJRT_Error* Buffer_OnDeviceSizeInBytes(
    PJRT_Buffer_OnDeviceSizeInBytes_Args* args) {
  args->on_device_size_in_bytes = args->buffer->size_bytes;
  return NULL;
}

static PJRT_Error* Buffer_Device(PJRT_Buffer_Device_Args* args) {
  args->device = args->buffer->device;
  return NULL;
}

static PJRT_Error* Buffer_Memory(PJRT_Buffer_Memory_Args* args) {
  args->memory = args->buffer->device->memory;
  return NULL;
}

static PJRT_Error* Buffer_Delete(PJRT_Buffer_Delete_Args* args) {
  args->buffer->deleted = true;
  return NULL;
}

static PJRT_Error* Buffer_IsDeleted(PJRT_Buffer_IsDeleted_Args* args) {
  args->is_deleted = args->buffer->deleted;
  return NULL;
}

static PJRT_Error* Buffer_IsOnCpu(PJRT_Buffer_IsOnCpu_Args* args) {
  args->is_on_cpu = false;  // the polite fiction that makes it a device
  return NULL;
}

static PJRT_Error* Buffer_ReadyEvent(PJRT_Buffer_ReadyEvent_Args* args) {
  args->event = make_ready_event();
  return NULL;
}

static PJRT_Error* Buffer_ToHostBuffer(PJRT_Buffer_ToHostBuffer_Args* args) {
  struct PJRT_Buffer* buf = args->src;
  if (args->dst == NULL) {
    // First call: the framework asks how much space it needs.
    args->dst_size = buf->size_bytes;
    return NULL;
  }
  if (args->dst_size < buf->size_bytes)
    return make_error(PJRT_Error_Code_INVALID_ARGUMENT,
                      "mock PJRT plugin: destination too small");
  memcpy(args->dst, buf->data, buf->size_bytes);
  args->event = make_ready_event();
  return NULL;
}

static PJRT_Error* Buffer_CopyToDevice(PJRT_Buffer_CopyToDevice_Args* args) {
  (void)args;
  return UNIMPLEMENTED("PJRT_Buffer_CopyToDevice");
}

static PJRT_Error* Buffer_CopyToMemory(PJRT_Buffer_CopyToMemory_Args* args) {
  (void)args;
  return UNIMPLEMENTED("PJRT_Buffer_CopyToMemory");
}

// ---------------------------------------------------------------- execute ---
// The refusal that makes this a mock. The error code and message travel the
// same path a real failure would: C error -> PjRtCApiClient -> Python.

static PJRT_Error* LoadedExecutable_Execute(
    PJRT_LoadedExecutable_Execute_Args* args) {
  (void)args;
  return UNIMPLEMENTED("PJRT_LoadedExecutable_Execute");
}

// ------------------------------------------------------------------ table ---

static PJRT_Api mock_api;

const PJRT_Api* GetPjrtApi(void) {
  init_topology();
  memset(&mock_api, 0, sizeof(mock_api));
  mock_api.struct_size = PJRT_Api_STRUCT_SIZE;
  mock_api.pjrt_api_version.struct_size = PJRT_Api_Version_STRUCT_SIZE;
  mock_api.pjrt_api_version.major_version = PJRT_API_MAJOR;
  mock_api.pjrt_api_version.minor_version = PJRT_API_MINOR;

  mock_api.PJRT_Error_Destroy = Error_Destroy;
  mock_api.PJRT_Error_Message = Error_Message;
  mock_api.PJRT_Error_GetCode = Error_GetCode;

  mock_api.PJRT_Plugin_Initialize = Plugin_Initialize;
  mock_api.PJRT_Plugin_Attributes = Plugin_Attributes;

  mock_api.PJRT_Client_Create = Client_Create;
  mock_api.PJRT_Client_Destroy = Client_Destroy;
  mock_api.PJRT_Client_PlatformName = Client_PlatformName;
  mock_api.PJRT_Client_ProcessIndex = Client_ProcessIndex;
  mock_api.PJRT_Client_PlatformVersion = Client_PlatformVersion;
  mock_api.PJRT_Client_Devices = Client_Devices;
  mock_api.PJRT_Client_AddressableDevices = Client_AddressableDevices;
  mock_api.PJRT_Client_LookupDevice = Client_LookupDevice;
  mock_api.PJRT_Client_LookupAddressableDevice = Client_LookupAddressableDevice;
  mock_api.PJRT_Client_AddressableMemories = Client_AddressableMemories;
  mock_api.PJRT_Client_DefaultDeviceAssignment = Client_DefaultDeviceAssignment;
  mock_api.PJRT_Client_TopologyDescription = Client_TopologyDescription;
  mock_api.PJRT_Client_Compile = Client_Compile;
  mock_api.PJRT_Client_BufferFromHostBuffer = Client_BufferFromHostBuffer;

  mock_api.PJRT_DeviceDescription_Id = DeviceDescription_Id;
  mock_api.PJRT_DeviceDescription_ProcessIndex = DeviceDescription_ProcessIndex;
  mock_api.PJRT_DeviceDescription_Attributes = DeviceDescription_Attributes;
  mock_api.PJRT_DeviceDescription_Kind = DeviceDescription_Kind;
  mock_api.PJRT_DeviceDescription_DebugString = DeviceDescription_DebugString;
  mock_api.PJRT_DeviceDescription_ToString = DeviceDescription_ToString;

  mock_api.PJRT_Device_GetDescription = Device_GetDescription;
  mock_api.PJRT_Device_IsAddressable = Device_IsAddressable;
  mock_api.PJRT_Device_LocalHardwareId = Device_LocalHardwareId;
  mock_api.PJRT_Device_AddressableMemories = Device_AddressableMemories;
  mock_api.PJRT_Device_DefaultMemory = Device_DefaultMemory;
  mock_api.PJRT_Device_MemoryStats = Device_MemoryStats;

  mock_api.PJRT_Memory_Id = Memory_Id;
  mock_api.PJRT_Memory_Kind = Memory_Kind;
  mock_api.PJRT_Memory_Kind_Id = Memory_KindId;
  mock_api.PJRT_Memory_DebugString = Memory_DebugString;
  mock_api.PJRT_Memory_ToString = Memory_ToString;
  mock_api.PJRT_Memory_AddressableByDevices = Memory_AddressableByDevices;

  mock_api.PJRT_Executable_Destroy = Executable_Destroy;
  mock_api.PJRT_Executable_Name = Executable_Name;
  mock_api.PJRT_Executable_NumReplicas = Executable_NumReplicas;
  mock_api.PJRT_Executable_NumPartitions = Executable_NumPartitions;
  mock_api.PJRT_Executable_NumOutputs = Executable_NumOutputs;
  mock_api.PJRT_Executable_SizeOfGeneratedCodeInBytes =
      Executable_SizeOfGeneratedCodeInBytes;
  mock_api.PJRT_Executable_GetCostAnalysis = Executable_GetCostAnalysis;
  mock_api.PJRT_Executable_OutputMemoryKinds = Executable_OutputMemoryKinds;
  mock_api.PJRT_Executable_OutputElementTypes = Executable_OutputElementTypes;
  mock_api.PJRT_Executable_OutputDimensions = Executable_OutputDimensions;
  mock_api.PJRT_Executable_OptimizedProgram = Executable_OptimizedProgram;
  mock_api.PJRT_Executable_Fingerprint = Executable_Fingerprint;

  mock_api.PJRT_Event_Destroy = Event_Destroy;
  mock_api.PJRT_Event_IsReady = Event_IsReady;
  mock_api.PJRT_Event_Error = Event_Error;
  mock_api.PJRT_Event_Await = Event_Await;
  mock_api.PJRT_Event_OnReady = Event_OnReady;

  mock_api.PJRT_Buffer_Destroy = Buffer_Destroy;
  mock_api.PJRT_Buffer_ElementType = Buffer_ElementType;
  mock_api.PJRT_Buffer_Dimensions = Buffer_Dimensions;
  mock_api.PJRT_Buffer_UnpaddedDimensions = Buffer_UnpaddedDimensions;
  mock_api.PJRT_Buffer_DynamicDimensionIndices = Buffer_DynamicDimensionIndices;
  mock_api.PJRT_Buffer_GetMemoryLayout = Buffer_GetMemoryLayout;
  mock_api.PJRT_Buffer_OnDeviceSizeInBytes = Buffer_OnDeviceSizeInBytes;
  mock_api.PJRT_Buffer_Device = Buffer_Device;
  mock_api.PJRT_Buffer_Memory = Buffer_Memory;
  mock_api.PJRT_Buffer_Delete = Buffer_Delete;
  mock_api.PJRT_Buffer_IsDeleted = Buffer_IsDeleted;
  mock_api.PJRT_Buffer_IsOnCpu = Buffer_IsOnCpu;
  mock_api.PJRT_Buffer_ReadyEvent = Buffer_ReadyEvent;
  mock_api.PJRT_Buffer_ToHostBuffer = Buffer_ToHostBuffer;
  mock_api.PJRT_Buffer_CopyToDevice = Buffer_CopyToDevice;
  mock_api.PJRT_Buffer_CopyToMemory = Buffer_CopyToMemory;

  mock_api.PJRT_LoadedExecutable_Destroy = LoadedExecutable_Destroy;
  mock_api.PJRT_LoadedExecutable_GetExecutable = LoadedExecutable_GetExecutable;
  mock_api.PJRT_LoadedExecutable_AddressableDevices =
      LoadedExecutable_AddressableDevices;
  mock_api.PJRT_LoadedExecutable_Delete = LoadedExecutable_Delete;
  mock_api.PJRT_LoadedExecutable_IsDeleted = LoadedExecutable_IsDeleted;
  mock_api.PJRT_LoadedExecutable_Execute = LoadedExecutable_Execute;

  return &mock_api;
}
'''
open("mock_gpu.c", "w").write(mock_gpu_c)
print(len(mock_gpu_c.splitlines()), "lines of C")

In [ ]:
!cc -shared -fPIC -I include mock_gpu.c -o mock_gpu.so && ls -la mock_gpu.so

In [ ]:
import os
os.environ["JAX_PLATFORMS"] = "mockgpu"

from jax._src import xla_bridge as xb
xb.register_plugin("mockgpu", library_path=os.path.abspath("mock_gpu.so"))

import jax
print(jax.devices())

Eight devices, each a struct in your C. Now the chapter 11 lesson, made touchable: one sharded `jax.Array` above, a pile of per-device buffers below.

In [ ]:
import numpy as np
import jax.numpy as jnp
from jax.sharding import Mesh, PartitionSpec as P, NamedSharding

mesh = Mesh(np.array(jax.devices()).reshape(8), ("d",))
sharded = NamedSharding(mesh, P("d"))
replicated = NamedSharding(mesh, P())

x = jax.device_put(np.arange(32, dtype=np.float32).reshape(8, 4), sharded)
print("one jax.Array:", x.shape, "on", len(x.sharding.device_set), "devices")
for s in x.addressable_shards[:3]:
    print("  shard on", s.device, "shape", s.data.shape)
print("  ...")

Eight `PJRT_Client_BufferFromHostBuffer` calls, eight `malloc`s, one per device, each holding a `(1, 4)` slice. The plugin never learns that the eight buffers belong together; PJRT has no concept of a sharded array. The single `jax.Array` you printed is IFRT's bookkeeping, one layer up, exactly as chapter 11 drew it.

**your prediction:** a `shard_map` program that calls `jax.lax.psum` compiles through a plugin whose compiler is the identity function. Will the all-reduce be visible in the bytes the plugin stores? Where did it come from, if no compiler pass ran?

In [ ]:
from jax.experimental.shard_map import shard_map

def local_sum_then_psum(x):
    return jax.lax.psum(x.sum(), axis_name="d")

f = shard_map(local_sum_then_psum, mesh=mesh, in_specs=P("d"), out_specs=P())
spec = jax.ShapeDtypeStruct((8, 4), jnp.float32)
lowered = jax.jit(f, in_shardings=sharded, out_shardings=replicated).lower(spec)

for line in lowered.as_text().splitlines():
    if "all_reduce" in line:
        print(line.strip()[:150])
        break

There it is, before the plugin ever sees it: `stablehlo.all_reduce`, with a `channel_handle` and `replica_groups` naming all eight devices. `shard_map` is the manual-collectives API, so JAX itself lowered your `psum` into the collective op during tracing. The collective is an instruction in the program, with the participating devices spelled out in its attributes. Nothing about it is an API call on the runtime.

Now push it across the seam.

In [ ]:
compiled = lowered.compile()
print("compiled:", type(compiled).__name__)

for line in compiled.as_text().splitlines():
    if "all-reduce" in line:
        print(line.strip()[:150])
        break

try:
    compiled(x)
except Exception as e:
    print("\n", str(e)[:170])

Read the three lines together and you have the whole collectives story. The stored program still carries `all-reduce` with `channel_id=1` and `replica_groups={{0,...,7}}`: your Compile was the identity, so the op passed through untouched. The execution attempt handed eight per-device buffer lists to `PJRT_LoadedExecutable_Execute`, chapter 14's nested-list signature doing exactly what it promised, and your C refused.

That refusal is standing in the exact spot where collectives are implemented for real. On a real GPU backend, chapter 9's codegen lowers the all-reduce into a collective thunk; at execute time that thunk calls NCCL through the stream machinery, using a communicator whose setup is the plugin's private business. The only place the PJRT interface acknowledges any of this is bootstrap: `PJRT_Client_Create_Args` carries key-value store callbacks (`PJRT_KeyValueGetCallback`, `PJRT_KeyValuePutCallback`) so a multi-host plugin can exchange NCCL ids between processes before the first collective runs. Compiler inserts the op, backend lowers it to a thunk, Execute runs it, the interface only carries the bytes and the bootstrap. That is the entire division of labor.

**your prediction:** replace `shard_map` with plain `jit` and sharding annotations, the automatic path from chapter 7. Does the all-reduce appear in the program JAX hands the plugin now?

In [ ]:
auto = jax.jit(lambda x: x.sum(), in_shardings=sharded, out_shardings=replicated).lower(spec)
print("all_reduce in the program JAX hands over?", "all_reduce" in auto.as_text())
for line in auto.as_text().splitlines():
    if "mhlo.sharding" in line:
        print("what is there instead:", line.strip()[:100])
        break
print("all-reduce after the mock 'compiled' it?  ", "all-reduce" in auto.compile().as_text())

No collective anywhere, before or after. The automatic path hands the plugin a program for one giant logical array, carrying only `mhlo.sharding` annotations, and trusts the backend compiler to run chapter 7's `SpmdPartitioner` and insert whatever collectives the math requires. Our mock skipped that rewrite, so the collective that a real backend would have manufactured simply never comes into existence. Run the same two lowerings on a real CPU or GPU backend and diff: `shard_map`'s all-reduce arrives from above, the annotated program's is born inside the compiler. Same op, two birthplaces, one runtime contract.

One honest scar from building this lab: with output shardings left unspecified, this compile fails against the mock, and the failure is informative. When sharding propagation is enabled, the framework asks the *compiled* executable for its per-shard output shapes (`GetOutputElementTypes`, `GetOutputDimensions` in `xla/python/pjrt_ifrt/pjrt_executable.cc`), because after the SPMD rewrite only the compiler knows them. A mock that skipped the rewrite cannot answer, which is why every `jit` in this notebook pins `out_shardings` explicitly: that keeps the framework on the path where it derives shapes from the input module. The interface trusts the compiler to know things the program alone does not say; the mock marks precisely where that trust begins.

## mark it run

Chapter 8 (kernels.rudrite.com/xla/collectives) is the reading this lab grounds: collectives as compiler-made instructions with device groups in their attributes, executed by the backend, invisible to the runtime interface. Chapter 7 explains the partitioner the automatic path trusts; chapter 14 explains every function your C implemented.

Provenance: verified 2026-08-10 against jax 0.4.38 / jaxlib 0.4.38 with the header at XLA commit `20a4825` (C API 0.58), on CPU (macOS arm64; the same pinned wheels ship for Colab's linux x86_64). Every output above reproduces by running this notebook top to bottom.